# 02 - 技术指标与可视化

本 Notebook 演示：
1. 手写实现常用技术指标
2. 绘制 K 线图
3. 叠加 MACD、RSI、KDJ、布林带
4. 综合技术分析仪表盘

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 设置中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

from src.data.storage import load_from_parquet
from src.indicators.calculator import *
from src.visualization.kline_chart import *
from src.visualization.overlay import *

print('Modules loaded')

## 1. 加载数据

In [ ]:
# 加载茅台数据
symbol = '600519'
df = load_from_parquet(symbol)
print(f'Rows: {len(df)}')
print(f'Date range: {df["date"].min()} ~ {df["date"].max()}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 2. 计算所有技术指标

In [ ]:
# 批量计算所有指标
df = add_all_indicators(df)
print(f'Total columns: {len(df.columns)}')

# 查看新增的指标列
indicator_cols = [c for c in df.columns if c not in ['date','open','close','high','low','volume','amount','turnover','outstanding_share','daily_return','log_return','cumulative_return','volume_lots','is_suspended']]
print(f'Indicators: {indicator_cols}')
df[['date','close'] + indicator_cols].tail()

## 3. K线图 (Candlestick Chart)

In [ ]:
# 绘制最近 120 天的 K 线图
recent = df.tail(120).copy()
plot_candlestick(recent, title=f'{symbol} Daily K-line', volume=True)

## 4. 移动平均线 (MA)

In [ ]:
# 金叉/死叉示例
fig, ax = plt.subplots(figsize=(14, 6))
subset = df.tail(120)
ax.plot(subset['date'], subset['close'], color='black', linewidth=0.8, label='Close')
ax.plot(subset['date'], subset['ma_5'], color='blue', linewidth=0.8, alpha=0.6, label='MA5')
ax.plot(subset['date'], subset['ma_20'], color='purple', linewidth=0.8, alpha=0.6, label='MA20')
ax.set_title('MA5 vs MA20 - Golden Cross / Death Cross')
ax.set_ylabel('Price (CNY)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 5. MACD 指标

In [ ]:
plot_macd(df.tail(120), save_name='macd_demo.png')

## 6. RSI 指标

In [ ]:
plot_rsi(df.tail(120))

## 7. KDJ 指标

In [ ]:
plot_kdj(df.tail(120))

## 8. 布林带 (Bollinger Bands)

In [ ]:
plot_bollinger(df.tail(120))

## 9. 综合技术分析仪表盘

In [ ]:
plot_full_dashboard(df.tail(120), title=f'{symbol} Technical Dashboard', save_name='dashboard_demo.png')

## 10. 指标学习笔记

### 金叉 (Golden Cross) 与 死叉 (Death Cross)
- **金叉**: 短期均线（如 MA5）从下方向上穿过长期均线（如 MA20）→ 买入信号
- **死叉**: 短期均线从上方向下穿过长期均线 → 卖出信号

### MACD 用法
- DIF 上穿 DEA → 金叉买入
- DIF 下穿 DEA → 死叉卖出
- MACD 柱由负转正 → 多头力量增强

### RSI 用法
- RSI > 70 → 超买 (Overbought)，可能回调
- RSI < 30 → 超卖 (Oversold)，可能反弹

### KDJ 用法
- K/D < 20 且 K 上穿 D → 买入信号
- K/D > 80 且 K 下穿 D → 卖出信号
- J > 100 超买，J < 0 超卖

### 布林带用法
- 价格触及上轨 → 可能回调
- 价格触及下轨 → 可能反弹
- 带宽收窄 → 可能即将变盘
- 带宽扩张 → 趋势加强